# Chunk evaluation

Read the published Notebook 05 handoff in a fresh project kernel. Notebook 06 starts with
the two existing paragraph plans, measured on the same parents. Counts and limits in this
initial comparison use Unicode code points. The analysis frame carries source, format, year,
length, grouping, and the inherited overlap evidence. Known overlap records have a separate
diagnostic view.

In [ ]:
import hashlib
import json
import os
import re
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "configs/preprocessing_notebook.json").is_file()
    ),
    None,
)
if ROOT is None or Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    raise RuntimeError("Open the project workspace and select its registered kernel.")

config = json.loads((ROOT / "configs/preprocessing_notebook.json").read_text())
profile = os.environ.get("FOT06_PROFILE", config["default_profile"])
resource = config["profiles"][profile]
output_root = (ROOT / config["output_directory"]).resolve()
HANDOFF = Path(os.environ.get("FOT06_INPUT", output_root / "formal_layout_step_v1_handoff_v1"))
HANDOFF = HANDOFF.resolve()
if HANDOFF.parent != output_root:
    raise ValueError("Select a versioned Notebook 05 handoff under the preprocessing root.")

if SparkSession.getActiveSession() is not None:
    raise RuntimeError("Start Notebook 06 in a fresh kernel.")
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS", f"--driver-memory {resource['driver_memory']} pyspark-shell"
)
spark = (
    SparkSession.builder.master(resource["master"])
    .appName("fear-temperature-chunk-evaluation")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", resource["shuffle_partitions"])
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024**2))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
if spark.version != config["spark_version"] or spark.sparkContext.master != resource["master"]:
    raise RuntimeError("Spark runtime differs from the selected project profile.")
print(f"Spark {spark.version} · {spark.sparkContext.master} · Notebook 06")

## Verify the input and define the comparison frame

Verify the manifest, closure receipt and Parquet file inventory. The release group is the
sampling unit for later boundary review. Keep the comparison and overlap diagnostic cohorts
available separately. Status without a match refers to the scope of the bound cross-split
audit.

In [ ]:
def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


handoff_manifest_path = HANDOFF / "MANIFEST.json"
closure_path = HANDOFF / "05_CLOSURE.json"
manifest = json.loads(handoff_manifest_path.read_text())
closure = json.loads(closure_path.read_text())
if (
    manifest["schema_version"] != 1
    or sha256_file(closure_path) != manifest["audit_sha256"]
    or closure["status"] != "verified_text_and_baseline_plans_for_chunk_evaluation"
):
    raise ValueError("Notebook 06 requires a verified Notebook 05 text handoff.")

expected = closure["checks"]["parent_rows"]
targets = manifest["targets_codepoints"]
if (
    expected != 718_330
    or manifest["targets_codepoints"] != [1200, 2400]
    or any(
        closure["checks"][field] != expected
        for field in (
            "unique_release_rows",
            "unique_native_identities",
            "verified_body_hashes",
            "parents_with_valid_plans",
            "parents_with_valid_paragraphs",
        )
    )
    or closure["step1_new_cross_split_exact_pairs"] != 0
    or closure["step2_new_cross_split_exact_pairs"] != 0
):
    raise ValueError("The 05 handoff has unresolved roster or baseline coverage checks.")

records_dir = HANDOFF / "records"
parts = manifest["tables"]["records"]
inventory = {path.relative_to(HANDOFF).as_posix(): path for path in records_dir.glob("*.parquet")}
if (
    not (records_dir / "_SUCCESS").is_file()
    or set(inventory) != {part["path"] for part in parts}
    or sum(part["rows"] for part in parts) != expected
):
    raise ValueError("The handoff Parquet inventory is incomplete.")
for part in parts:
    file = inventory[part["path"]]
    if (
        sha256_file(file) != part["sha256"]
        or pq.ParquetFile(file).metadata.num_rows != part["rows"]
    ):
        raise ValueError(f"The handoff Parquet part failed verification: {part['path']}")

parents = spark.read.parquet(str(records_dir))
strata = parents.select(
    "release_row",
    "native_identity",
    "release_group_id",
    "ml_split",
    "source_id",
    "source_host",
    "format",
    "publication_year",
    "publication_precision",
    "clean_sha256",
    "clean_chars",
    "overlap_status",
    "overlap_pair_ids",
    "cross_split_review_required",
    "plans",
).withColumn(
    "length_band",
    F.when(F.col("clean_chars") <= 250, "00001-00250")
    .when(F.col("clean_chars") <= 1200, "00251-01200")
    .when(F.col("clean_chars") <= 2400, "01201-02400")
    .when(F.col("clean_chars") <= 5000, "02401-05000")
    .when(F.col("clean_chars") <= 10000, "05001-10000")
    .otherwise("10001+"),
)
comparison_parents = strata.where(~F.col("cross_split_review_required"))
overlap_diagnostics = strata.where(F.col("cross_split_review_required"))
counts = [
    row.asDict()
    for row in strata.groupBy("ml_split", "overlap_status")
    .count()
    .orderBy("ml_split", "overlap_status")
    .collect()
]
if counts != closure["overlap_status_by_split"]:
    raise ValueError("The evaluation strata disagree with the bound overlap counts.")

group_conflicts = (
    strata.groupBy("release_group_id")
    .agg(F.countDistinct("ml_split").alias("split_count"))
    .where(F.col("split_count") > 1)
    .limit(1)
    .count()
)
if group_conflicts or strata.where(F.col("release_group_id").isNull()).limit(1).count():
    raise ValueError("A release group is missing or crosses the inherited split.")

print(f"Verified {expected:,} parents and {len(parts)} handoff files.")
print(f"Notebook 05 handoff manifest SHA-256: {sha256_file(handoff_manifest_path)}")
display(pd.DataFrame(counts))

## Measure the existing paragraph plans

Measure the 1,200 and 2,400 code-point baselines, including single-chunk rates and chunks that
exceed the target. The `baseline_by_stratum` view keeps the format, length and overlap cohorts
available for the next comparison.

In [ ]:
plan_rows = (
    strata.select(
        "release_row",
        "format",
        "length_band",
        "cross_split_review_required",
        F.explode("plans").alias("plan"),
    )
    .select(
        "release_row",
        "format",
        "length_band",
        "cross_split_review_required",
        F.col("plan.target").alias("target"),
        F.col("plan.ends").alias("ends"),
    )
    .withColumn(
        "chunk_lengths",
        F.zip_with(
            "ends",
            F.slice(F.concat(F.array(F.lit(0)), "ends"), 1, F.size("ends")),
            lambda end, start: end - start,
        ),
    )
    .withColumn("chunks", F.size("ends"))
    .withColumn(
        "oversized_chunks", F.size(F.filter("chunk_lengths", lambda n: n > F.col("target")))
    )
)
baseline_by_stratum = plan_rows.groupBy(
    "target", "format", "length_band", "cross_split_review_required"
).agg(
    F.count("*").alias("parents"),
    F.sum("chunks").alias("chunks"),
    F.count_if(F.col("chunks") == 1).alias("one_chunk_parents"),
    F.sum("oversized_chunks").alias("chunks_over_target"),
    F.count_if(F.col("oversized_chunks") > 0).alias("parents_with_chunk_over_target"),
)
baseline_totals = (
    baseline_by_stratum.groupBy("target")
    .agg(
        F.sum("parents").alias("parents"),
        F.sum("chunks").alias("chunks"),
        F.sum("one_chunk_parents").alias("one_chunk_parents"),
        F.sum("chunks_over_target").alias("chunks_over_target"),
        F.sum("parents_with_chunk_over_target").alias("parents_with_chunk_over_target"),
    )
    .orderBy("target")
    .collect()
)
if (
    [row.target for row in baseline_totals] != targets
    or any(row.parents != expected for row in baseline_totals)
    or [{"target": row.target, "chunks": row.chunks} for row in baseline_totals]
    != closure["final_baseline_chunks"]
):
    raise ValueError("The measured baselines do not match the verified handoff.")
display(pd.DataFrame([row.asDict() for row in baseline_totals]))
print("Measured both existing code-point plans on the identical parent roster.")

## Check exact chunk reuse across splits

Hash exact slices of the verified text. Report distinct hash buckets spanning two or three
inherited splits and the total number of occurrences in those buckets. A chunk can match
across splits even when its complete parent body is distinct. Report all lengths and a
separate diagnostic at 500 code points. The comparison columns recompute the counts with
the known parent-overlap diagnostic cohort held aside. Repeated lexical content and shared boilerplate can
both produce a match. The hash buckets define a review set.

In [ ]:
chunk_occurrences = (
    parents.select(
        "release_row",
        "ml_split",
        "cross_split_review_required",
        "clean_text",
        F.explode("plans").alias("plan"),
    )
    .select(
        "release_row",
        "ml_split",
        "cross_split_review_required",
        "clean_text",
        F.col("plan.target").alias("target"),
        F.col("plan.ends").alias("ends"),
        F.posexplode("plan.ends").alias("ordinal", "end"),
    )
    .withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
    .withColumn("chunk_codepoints", F.col("end") - F.col("start"))
    .withColumn(
        "chunk_sha256",
        F.sha2(F.substring("clean_text", F.col("start") + 1, F.col("chunk_codepoints")), 256),
    )
    .drop("clean_text", "ends")
)
# Each bit represents one inherited split. A mask above one bit spans splits.
split_bit = (
    F.when(F.col("ml_split") == "train", 1).when(F.col("ml_split") == "validation", 2).otherwise(4)
)
in_comparison = ~F.col("cross_split_review_required")
cross_split_chunk_reuse = (
    chunk_occurrences.groupBy("target", "chunk_sha256", "chunk_codepoints")
    .agg(
        F.bit_or(split_bit).alias("split_mask"),
        F.bit_or(F.when(in_comparison, split_bit).otherwise(0)).alias("comparison_split_mask"),
        F.count("*").alias("chunk_occurrences"),
        F.count_if(in_comparison).alias("comparison_occurrences"),
    )
    .where(F.col("split_mask").isin(3, 5, 6, 7))
)
crosses_comparison = F.col("comparison_split_mask").isin(3, 5, 6, 7)
reuse_counts = (
    cross_split_chunk_reuse.withColumn(
        "minimum_codepoints",
        F.explode(
            F.when(F.col("chunk_codepoints") >= 500, F.array(F.lit(0), F.lit(500))).otherwise(
                F.array(F.lit(0))
            )
        ),
    )
    .groupBy("target", "minimum_codepoints")
    .agg(
        F.count("*").alias("cross_split_hash_buckets"),
        F.sum("chunk_occurrences").alias("chunk_occurrences"),
        F.count_if(crosses_comparison).alias("comparison_hash_buckets"),
        F.sum(F.when(crosses_comparison, F.col("comparison_occurrences")).otherwise(0)).alias(
            "comparison_chunk_occurrences"
        ),
    )
)
reuse_report = (
    spark.createDataFrame(
        [(target, cutoff) for target in targets for cutoff in (0, 500)],
        ["target", "minimum_codepoints"],
    )
    .join(reuse_counts, ["target", "minimum_codepoints"], "left")
    .fillna(0)
    .orderBy("target", "minimum_codepoints")
    .collect()
)
display(pd.DataFrame([row.asDict() for row in reuse_report]))
print("Measured exact chunk reuse across splits by baseline and length.")

## Prepare the full review queue for complex text

Queue every parent in the declared publication formats or designated source IDs. Include
long bodies and parents with oversized chunks, retained formatting markers, upstream review
evidence, or known overlap in any format. Record every reason. The source IDs in
`FOT06_COMPLEX_SOURCE_IDS` extend the scope to complete sources. Pin the scope and input
version before starting review. Set `FOT06_REVIEW_OUTPUT` to write an alternate scope.

The queue binds the verified body and its complete 1,200-code-point baseline plan. Each
queued parent receives consecutive review windows covering the whole body. Neighbouring
context accompanies each window. Review window boundaries only assign work and carry no
authority as chunk boundaries. This snapshot measures the review workload. A later
tokenizer-aware plan requires its own bound review packets.


In [ ]:
COMPLEX_FORMATS = (
    "Article",
    "Blog",
    "Community publication",
    "Literary",
    "News",
    "Newsletter",
    "Other",
    "Public submission",
    "Transcript",
    "Unresolved",
)
DESIGNATED_SOURCE_IDS = sorted(
    {
        source.strip()
        for source in os.environ.get("FOT06_COMPLEX_SOURCE_IDS", "").split(",")
        if source.strip()
    }
)
REVIEW_CORE_CODEPOINTS = 4_800
REVIEW_CONTEXT_CODEPOINTS = 320
REVIEW_BASELINE_TARGET = 1_200
REVIEW_POLICY = "complex_boundary_review_v1"
near_delta_path = HANDOFF.with_name(f"{HANDOFF.name}_near_delta_v1")
near_manifest_path = near_delta_path / "MANIFEST.json"
near_manifest = json.loads(near_manifest_path.read_text())
near_audit_path = near_delta_path / "AUDIT.json"
if near_manifest["input_handoff_manifest_sha256"] != sha256_file(
    handoff_manifest_path
) or near_manifest["audit_sha256"] != sha256_file(near_audit_path):
    raise ValueError("The incremental near-duplicate audit does not match this handoff.")


def bound_review_table(root, name, entries):
    directory = root / name
    inventory = {p.relative_to(root).as_posix(): p for p in directory.glob("*.parquet")}
    if not (directory / "_SUCCESS").is_file() or set(inventory) != {
        entry["path"] for entry in entries
    }:
        raise ValueError(f"Incomplete review table {name}.")
    for entry in entries:
        p = inventory[entry["path"]]
        if (
            sha256_file(p) != entry["sha256"]
            or pq.ParquetFile(p).metadata.num_rows != entry["rows"]
        ):
            raise ValueError(f"The review table failed verification: {entry['path']}")
    return spark.read.parquet(str(directory))


near_pairs = bound_review_table(
    near_delta_path, "cross_split_candidates", near_manifest["tables"]["cross_split_candidates"]
)
near_parents = (
    near_pairs.where(F.col("after.match"))
    .select(F.explode(F.array("row_a", "row_b")).alias("release_row"))
    .distinct()
    .withColumn("near_delta_match", F.lit(True))
)
default_queue_dir = HANDOFF.with_name(f"{HANDOFF.name}_chunk_review_v1")
queue_dir = Path(os.environ.get("FOT06_REVIEW_OUTPUT", default_queue_dir)).resolve()
if queue_dir.parent != HANDOFF.parent or queue_dir in (HANDOFF, near_delta_path):
    raise ValueError("Select a separate queue directory under the preprocessing root.")
queue_contract = {
    "review_policy": REVIEW_POLICY,
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
    "near_delta_manifest_sha256": sha256_file(near_manifest_path),
    "complex_formats": list(COMPLEX_FORMATS),
    "designated_source_ids": DESIGNATED_SOURCE_IDS,
    "baseline_target_codepoints": REVIEW_BASELINE_TARGET,
    "review_core_codepoints": REVIEW_CORE_CODEPOINTS,
    "review_context_codepoints": REVIEW_CONTEXT_CODEPOINTS,
}

if queue_dir.exists():
    queue_manifest = json.loads((queue_dir / "MANIFEST.json").read_text())
    queue_audit_path = queue_dir / "AUDIT.json"
    if any(
        queue_manifest.get(key) != value for key, value in queue_contract.items()
    ) or queue_manifest["audit_sha256"] != sha256_file(queue_audit_path):
        raise ValueError("Use a new review version for changed sources, inputs or review rules.")
    review_queue = bound_review_table(queue_dir, "queue", queue_manifest["queue_files"])
    queue_audit = json.loads(queue_audit_path.read_text())
else:
    if DESIGNATED_SOURCE_IDS:
        actual_ids = {
            row.source_id
            for row in parents.where(F.col("source_id").isin(*DESIGNATED_SOURCE_IDS))
            .select("source_id")
            .distinct()
            .collect()
        }
        if actual_ids != set(DESIGNATED_SOURCE_IDS):
            raise ValueError("The designated review sources include IDs outside the formal roster.")

    baseline = F.try_element_at(
        F.filter("plans", lambda plan: plan.target == REVIEW_BASELINE_TARGET), F.lit(1)
    )
    review_population = (
        parents.select(
            "release_row",
            "native_identity",
            "release_group_id",
            "ml_split",
            "format",
            "source_id",
            "source_host",
            "clean_sha256",
            "clean_chars",
            "overlap_status",
            "cross_split_review_required",
            "retained_marker_counts",
            "structure_review_reasons",
            "first_pass_review_reasons",
            "layout_review_reasons",
            "plans",
        )
        .join(F.broadcast(near_parents), "release_row", "left")
        .withColumn("plan", baseline)
    )
    long_chunk = F.exists(
        F.zip_with(
            "plan.ends",
            F.slice(F.concat(F.array(F.lit(0)), "plan.ends"), 1, F.size("plan.ends")),
            lambda end, start: end - start,
        ),
        lambda chars: chars > REVIEW_BASELINE_TARGET,
    )
    reasons = {
        "complex_publication_format": F.col("format").isin(*COMPLEX_FORMATS),
        "designated_source": (
            F.col("source_id").isin(*DESIGNATED_SOURCE_IDS)
            if DESIGNATED_SOURCE_IDS
            else F.lit(False)
        ),
        "long_body": F.col("clean_chars") > 2400,
        "oversized_baseline_chunk": long_chunk,
        "retained_markup_or_spacing": F.size("retained_marker_counts") > 0,
        "upstream_review_evidence": (
            (F.size("structure_review_reasons") > 0)
            | (F.size("first_pass_review_reasons") > 0)
            | (F.size("layout_review_reasons") > 0)
        ),
        "known_parent_overlap": F.col("cross_split_review_required"),
        "incremental_near_duplicate_match": F.col("near_delta_match"),
    }
    review_population = review_population.withColumn(
        "review_reasons",
        F.filter(
            F.array(*[F.when(condition, F.lit(name)) for name, condition in reasons.items()]),
            lambda reason: reason.isNotNull(),
        ),
    )
    queue_fields = (
        "release_row",
        "native_identity",
        "release_group_id",
        "ml_split",
        "format",
        "source_id",
        "source_host",
        "clean_sha256",
        "clean_chars",
        "overlap_status",
        "review_reasons",
        "plan",
    )
    cores = F.sequence(F.lit(0), F.col("clean_chars") - 1, F.lit(REVIEW_CORE_CODEPOINTS))
    queue_source = (
        review_population.where(F.size("review_reasons") > 0)
        .select(*queue_fields)
        .withColumn("plan_sha256", F.sha2(F.to_json("plan"), 256))
        .withColumn(
            "review_units",
            F.transform(
                cores,
                lambda start, ordinal: F.struct(
                    ordinal.alias("unit_ordinal"),
                    start.alias("start"),
                    F.least(start + REVIEW_CORE_CODEPOINTS, "clean_chars").alias("end"),
                    F.greatest(F.lit(0), start - REVIEW_CONTEXT_CODEPOINTS).alias("context_start"),
                    F.least(
                        start + REVIEW_CORE_CODEPOINTS + REVIEW_CONTEXT_CODEPOINTS, "clean_chars"
                    ).alias("context_end"),
                ),
            ),
        )
    )

    with tempfile.TemporaryDirectory(prefix=".chunk-review-", dir=queue_dir.parent) as local:
        staged_queue = Path(local) / "snapshot"
        (
            queue_source.write.mode("errorifexists")
            .option("compression", "zstd")
            .parquet(str(staged_queue / "queue"))
        )
        new_queue = spark.read.parquet(str(staged_queue / "queue"))
        coverage = (
            new_queue.agg(
                F.count("*").alias("required_parents"),
                F.countDistinct("release_row").alias("distinct_parents"),
                F.sum(F.size("review_units")).alias("required_work_units"),
                F.count_if(
                    F.aggregate(
                        "review_units", F.lit(0), lambda total, unit: total + unit.end - unit.start
                    )
                    == F.col("clean_chars")
                ).alias("parents_with_full_review_windows"),
                F.count_if(F.col("plan_sha256").isNull()).alias("missing_plan_hashes"),
            )
            .first()
            .asDict()
        )
        expected_scope = review_population.groupBy(F.size("review_reasons") > 0).count().collect()
        population_counts = {row[0]: row["count"] for row in expected_scope}
        parent_count = population_counts.get(True, 0)
        if (
            sum(population_counts.values()) != expected
            or any(
                coverage[k] != parent_count
                for k in (
                    "required_parents",
                    "distinct_parents",
                    "parents_with_full_review_windows",
                )
            )
            or coverage["missing_plan_hashes"]
        ):
            raise ValueError("The review queue does not cover exactly the declared population.")
        by_format = [
            row.asDict()
            for row in new_queue.groupBy("format")
            .agg(
                F.count("*").alias("parents"),
                F.countDistinct("source_id").alias("source_ids"),
                F.sum(F.size("review_units")).alias("work_units"),
            )
            .orderBy("format")
            .collect()
        ]
        reason_counts = [
            row.asDict()
            for row in new_queue.select(F.explode("review_reasons").alias("reason"))
            .groupBy("reason")
            .count()
            .orderBy("reason")
            .collect()
        ]
        queue_audit = {
            "formal_parents": expected,
            "outside_declared_complex_scope": population_counts.get(False, 0),
            **coverage,
            "by_format": by_format,
            "reason_counts": reason_counts,
            "initial_status": "pending_external_or_local_reviewer_decisions",
        }
        queue_audit_path = staged_queue / "AUDIT.json"
        queue_audit_path.write_text(json.dumps(queue_audit, indent=2) + "\n")
        queue_files = [
            {
                "path": p.relative_to(staged_queue).as_posix(),
                "sha256": sha256_file(p),
                "rows": pq.ParquetFile(p).metadata.num_rows,
            }
            for p in sorted((staged_queue / "queue").glob("*.parquet"))
        ]
        queue_manifest = {
            **queue_contract,
            "audit_sha256": sha256_file(queue_audit_path),
            "queue_files": queue_files,
        }
        (staged_queue / "MANIFEST.json").write_text(json.dumps(queue_manifest, indent=2) + "\n")
        bound_review_table(staged_queue, "queue", queue_files)
        if queue_dir.exists():
            raise FileExistsError("The review queue destination appeared during publication.")
        staged_queue.rename(queue_dir)
    review_queue = spark.read.parquet(str(queue_dir / "queue"))

display(pd.DataFrame(queue_audit["by_format"]))
display(pd.DataFrame(queue_audit["reason_counts"]))
print(
    f"{queue_audit['required_parents']:,} parents require agent review in "
    f"{queue_audit['required_work_units']:,} work units."
)

## Verify the review packet and decision contracts

An agent reads each unit with its surrounding context and returns the packet hash,
reviewer and revision, outcome, reason and selected boundary IDs. Reviewers can accept the
baseline boundaries, adjust them using candidate IDs, or escalate. Candidate IDs refer to
positions in the immutable parent text. Inspect whether attribution, negation, quotation,
intensity and temperature references stay connected across each proposed boundary.

A parent is ready under the boundary review protocol after every current unit has one
accepted or adjusted decision and the offsets form a complete ordered plan. Missing or
escalated units remain open. A calibration batch can estimate effort and improve the
instructions. Completion is measured on the full declared queue. Token limits and cross-split
checks apply separately to an adopted plan.


In [ ]:
from pyspark.sql import Row

SENTENCE_END = re.compile(r"""[.!?。！？]+["'”’\)\]]{0,3}[ \t\n]+""")


def canonical_sha256(value):
    serialized = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def make_review_packets(queued_parent, body, paragraphs):
    parent_sha = hashlib.sha256(body.encode("utf-8")).hexdigest()
    if parent_sha != queued_parent.clean_sha256 or len(body) != queued_parent.clean_chars:
        raise ValueError("The review body does not match the queued parent.")
    baseline = queued_parent.plan.ends
    if baseline != sorted(set(baseline)) or baseline[-1] != len(body):
        raise ValueError("The parent baseline does not cover the body.")
    proposed = set(baseline[:-1])
    candidates = proposed | {p.start for p in paragraphs if p.start > 0}
    candidates |= {
        match.end() for match in SENTENCE_END.finditer(body) if 0 < match.end() < len(body)
    }
    packet_list = []
    cursor = 0
    for ordinal, unit in enumerate(queued_parent.review_units):
        if not (
            unit.unit_ordinal == ordinal
            and unit.start == cursor
            and 0 <= unit.context_start <= unit.start < unit.end <= unit.context_end <= len(body)
        ):
            raise ValueError("The review windows do not tile the parent body.")
        available = sorted(offset for offset in candidates if unit.start <= offset < unit.end)
        packet = {
            "review_policy": REVIEW_POLICY,
            "release_row": queued_parent.release_row,
            "body_sha256": parent_sha,
            "plan_sha256": queued_parent.plan_sha256,
            "unit_ordinal": ordinal,
            "start": unit.start,
            "end": unit.end,
            "context_start": unit.context_start,
            "context_end": unit.context_end,
            "baseline_boundary_ids": [offset for offset in available if offset in proposed],
            "candidate_boundary_ids": available,
            "context_text": body[unit.context_start : unit.context_end],
        }
        packet_list.append({**packet, "packet_sha256": canonical_sha256(packet)})
        cursor = unit.end
    if cursor != len(body):
        raise ValueError("The review packets do not cover the complete body.")
    return packet_list


def validate_reviewed_parent(queued_parent, body, paragraphs, decisions):
    packets = make_review_packets(queued_parent, body, paragraphs)
    by_hash = {packet["packet_sha256"]: packet for packet in packets}
    supplied = {}
    accepted_boundaries = []
    escalated = False
    for decision in decisions:
        digest = decision["packet_sha256"]
        if digest not in by_hash or digest in supplied:
            raise ValueError("The review decision refers to a stale or duplicate packet.")
        status = decision["status"]
        if status not in {"accept", "adjust", "escalate"} or any(
            not isinstance(decision.get(key), str) or not decision[key].strip()
            for key in ("reviewer", "reviewer_version", "reason")
        ):
            raise ValueError("The review decision is missing an outcome or provenance.")
        packet = by_hash[digest]
        ids = decision["selected_boundary_ids"]
        if ids != sorted(set(ids)) or not set(ids) <= set(packet["candidate_boundary_ids"]):
            raise ValueError("The review decision selects an unavailable boundary.")
        if status == "accept" and ids != packet["baseline_boundary_ids"]:
            raise ValueError("An accepted packet must preserve the proposed boundaries.")
        if status == "adjust" and ids == packet["baseline_boundary_ids"]:
            raise ValueError("An adjusted packet must change at least one proposed boundary.")
        if status == "escalate":
            if ids:
                raise ValueError("Escalate the packet before committing boundary choices.")
            escalated = True
        accepted_boundaries.extend(ids)
        supplied[digest] = status
    if escalated or len(supplied) != len(by_hash):
        return {
            "status": "escalated" if escalated else "pending",
            "required_units": len(by_hash),
            "received_units": len(supplied),
            "ends": None,
        }
    selected_ends = sorted(accepted_boundaries) + [len(body)]
    if selected_ends != sorted(set(selected_ends)) or not all(
        0 < end <= len(body) for end in selected_ends
    ):
        raise ValueError("The assembled reviewed plan must have distinct ordered boundaries.")
    return {
        "status": "boundary_review_complete",
        "required_units": len(by_hash),
        "received_units": len(supplied),
        "ends": selected_ends,
    }


# A UTF-8 body split into two review units exercises complete and incomplete acceptance.
example = "Not afraid at 40°C. Next sentence.\n\nAnother speaker worried."
fixture = Row(
    release_row=0,
    clean_sha256=hashlib.sha256(example.encode()).hexdigest(),
    clean_chars=len(example),
    plan=Row(ends=[20, len(example)]),
    plan_sha256="fixture-plan",
    review_units=[
        Row(unit_ordinal=0, start=0, end=28, context_start=0, context_end=40),
        Row(unit_ordinal=1, start=28, end=len(example), context_start=14, context_end=len(example)),
    ],
)
fixture_paragraphs = [Row(start=0), Row(start=36)]
fixture_packets = make_review_packets(fixture, example, fixture_paragraphs)
fixture_decisions = [
    {
        "packet_sha256": packet["packet_sha256"],
        "status": "accept",
        "reviewer": "fixture",
        "reviewer_version": "example_v1",
        "reason": "The proposed boundary preserves the evidence in the context.",
        "selected_boundary_ids": packet["baseline_boundary_ids"],
    }
    for packet in fixture_packets
]
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, fixture_decisions)[
    "ends"
] == (fixture.plan.ends)
assert (
    validate_reviewed_parent(fixture, example, fixture_paragraphs, fixture_decisions[:1])["status"]
    == "pending"
)
adjusted = [dict(row) for row in fixture_decisions]
adjusted[1].update(status="adjust", selected_boundary_ids=[36])
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, adjusted)["ends"] == [
    20,
    36,
    len(example),
]
escalated = [dict(row) for row in fixture_decisions]
escalated[1]["status"] = "escalate"
assert validate_reviewed_parent(fixture, example, fixture_paragraphs, escalated)["ends"] is None
for bad in (
    [*fixture_decisions, fixture_decisions[0]],
    [{**fixture_decisions[0], "packet_sha256": "wrong-packet"}, fixture_decisions[1]],
    [{**fixture_decisions[0], "selected_boundary_ids": [-1]}, fixture_decisions[1]],
):
    try:
        validate_reviewed_parent(fixture, example, fixture_paragraphs, bad)
    except ValueError:
        pass
    else:
        raise AssertionError("An invalid agent decision was accepted.")

probe = review_queue.where(F.size("review_units") > 1).orderBy("release_row").first()
if probe is not None:
    text_row = (
        parents.where(F.col("release_row") == probe.release_row)
        .select("clean_text", "paragraphs")
        .first()
    )
    packet_probe = make_review_packets(probe, text_row.clean_text, text_row.paragraphs)
    assert len(packet_probe) == len(probe.review_units)
    assert packet_probe[-1]["end"] == probe.clean_chars
print("Verified the review protocol on synthetic decisions and one real packet set.")
print("The published queue starts with every required work unit awaiting a reviewer decision.")

## Token-aware comparison and review execution

Select and pin a counting tokenizer for the next comparison. Record the revision and the
content budget before creating token-aware plans. Compare the two code-point baselines and
the candidate plan on the same release groups. Verify coverage and exact cross-split
chunk reuse for the chosen offsets. Report evidence-span retention, review revisions,
escalations, reviewer agreement, latency and cost. Keep reviewer decisions bound to the
plan and corpus versions that the reviewers received.

## Size the reviewer run and hand it off to Notebook 07

### Current planning measurement

Measured on 21 November 2026 against the 1,200-code-point review queue with manifest SHA-256
`80ec3a0a745bf5c810432b2fe6f6051e095fb64c6ab43f33c36eeea35863b526`. The queue has
385,749 parents and 663,265 review units. All context windows total 2,371,802,230 code points.
The offline measurement ranked units by a deterministic SHA-256 seed
`review-sizing-20261121`, sampled up to 120 units in each of 64 format and context-length
strata, and weighted the 6,666 sampled units back to the full queue. Context-length upper
bounds were 600, 1,200, 2,400, 4,000 and 5,600 code points.

| Quantity | Planning value |
|---|---:|
| Context under the reference encoding | 509.1 million tokens |
| Complete request input under the reference encoding | 847.4 million tokens |
| One request per unit as uncompressed JSONL | 3.40 GiB |
| Offline estimation peak process RSS | 405 MiB |

The input estimate uses `cl100k_base`, the complete packet, a proposed short instruction and
response schema, and a 24-token framing allowance. The sampling margin for the request
estimate is approximately 6.3 million reference tokens. Provider tokenizer and request
framing differences need a live usage measurement. This workload assumes one request per
unit and should be recomputed for the adopted boundary plan.

### Calibration in Notebook 06

Use the same proposed pilot of 240 stratified review units for GPT-6.2 Sol Medium and High.
Include short controls and difficult quotation, speaker attribution, negation and long-text
cases. Adjudicate semantic disagreements and inspect a blinded selection of agreements.
Measure valid receipts per elapsed hour, wall-clock latency, retry and escalation rates,
and actual input, output and reasoning tokens. Use Medium for the primary review if it
passes calibration and High for ambiguous cases and second-pass adjudication. Save the
reviewer model ID and reasoning effort in each accepted receipt.

For full-queue planning, 5,000, 15,000 and 30,000 final valid units per elapsed hour imply
approximately 133, 44 and 22 hours of production review respectively. These rates are
scenarios. Record the project's actual queued-token capacity and observe elapsed throughput
from completed batches before forecasting the end date. Allow additional calendar time for
calibration and unresolved escalations. A Batch completion window applies to each accepted
batch.

At the published Sol Batch rates of USD 1 per million input tokens and USD 5 per million
output tokens, the reference input alone is about USD 847. At 500 or 1,000 billed output
tokens per unit, including reasoning, illustrative total costs are about USD 2,506 and
USD 4,164 before retries and escalations. Use measured usage for a production budget.

[GPT-6.2 Sol](https://developers.openai.com/api/docs/models/gpt-6.2-sol) ·
[Batch limits and status](https://developers.openai.com/api/docs/guides/batch#rate-limits) ·
[Reasoning token accounting](https://developers.openai.com/api/docs/guides/reasoning#how-reasoning-works)

### Production in Notebook 07

Use 07 for the versioned, restartable batch controller and results. Read the frozen queue
and committed batch IDs. Prepare and upload bounded request shards, ingest results as
streams, and retry only unsettled unit IDs. Store the compact receipt once per validated
unit. An already submitted server-side batch can complete while the laptop is offline.
Local submission, reconciliation and new batches resume from the ledger when it reconnects.

Use a local ledger keyed by packet digest and an inspectable `RUN_STATUS.json` with the
queue version, UTC update time, units pending, submitted, ready, escalated or failed, fully
accepted parents, rolling valid units per hour, remaining time, token usage, cost and peak
RSS. Save batch IDs, file hashes and counts per batch. Save the reviewer model, prompt
version, outcome, chosen boundaries, short reason and usage per unit. Retain full raw
responses for invalid or disputed cases when useful. Recreate review windows from the
bound source text and offsets. A local review count increases only for a new valid
receipt. A parent is complete when every required unit has an approved boundary decision.

Verify manifests once on run initialization or a version change. For each receipt check
schema, packet and plan binding, boundary IDs and accepted-unit uniqueness. Verify final
coverage, tokenizer budget and exact cross-split chunk matches on the adopted plan once at
publication. Semantic second review is for the pilot, difficult cases and disagreements.
Use an approved processing environment for the selected corpus text.

The observed host is an Apple M1 with 16 GiB of unified memory. A streaming controller can
be planned with 4–6 GiB of usable memory and one active request shard near 64 MiB. Allow
6–8 GiB for a separate Spark pass with the project local[6] profile. Aim for 15–20 GiB
of free disk before production to cover active files, compact receipts and system swap.
At measurement time, approximately 5.5 GiB was available. Keep the production review
controller and heavy Spark jobs in separate runs.

## Freeze the paired calibration packets

Select complete parents from development release groups. Use deterministic draws balanced
across formats and sources within the four workload tiers. Keep all units for each selected
parent. The two Sol effort settings receive the same 240 packet digests. The pilot ledger
starts with all reviewer results open.

Save a compact inventory of packet IDs, parent IDs, provenance hashes and bounds.
The context can be reconstructed from the bound Notebook 05 handoff. The pilot inventory
contains no corpus excerpts. The live calibration will measure disagreement, valid receipt
throughput, latency and billed tokens on an approved reviewer route.


In [ ]:
from pyspark.sql import Window

PILOT_SEED = "paired-medium-high-pilot-v1"
PILOT_TARGET_UNITS = 240
PILOT_LANES = {
    "sol_medium": {"model": "gpt-6.2-sol", "effort": "medium"},
    "sol_high": {"model": "gpt-6.2-sol", "effort": "high"},
}
pilot_dir = queue_dir.with_name(f"{queue_dir.name}_calibration_v1")
pilot_contract = {
    "policy": PILOT_SEED,
    "queue_manifest_sha256": sha256_file(queue_dir / "MANIFEST.json"),
    "input_handoff_manifest_sha256": sha256_file(handoff_manifest_path),
    "near_delta_manifest_sha256": sha256_file(near_manifest_path),
    "target_units": PILOT_TARGET_UNITS,
    "lanes": PILOT_LANES,
}
pilot_manifest_path = pilot_dir / "MANIFEST.json"
if pilot_dir.exists():
    pilot_manifest = json.loads(pilot_manifest_path.read_text())
    if any(pilot_manifest.get(key) != value for key, value in pilot_contract.items()):
        raise ValueError("The saved pilot belongs to a different queue or paired design.")
    if sha256_file(pilot_dir / "PACKETS.json") != pilot_manifest["packets_sha256"]:
        raise ValueError("The pilot packet inventory failed verification.")
    pilot_inventory = json.loads((pilot_dir / "PACKETS.json").read_text())
else:
    review_evidence = F.exists(
        "review_reasons",
        lambda reason: reason.isin("retained_markup_or_spacing", "upstream_review_evidence"),
    )
    eligible = (
        review_queue.where(
            (F.col("ml_split") == "train")
            & (F.col("overlap_status") == "no_cross_split_evidence_in_current_audit")
            & ~F.array_contains("review_reasons", "incremental_near_duplicate_match")
        )
        .withColumn("units", F.size("review_units"))
        .withColumn(
            "tier",
            F.when(F.col("units") > 1, "multiple_windows")
            .when(review_evidence, "structural_review")
            .when(F.col("clean_chars") <= 1200, "short_control")
            .otherwise("longer_single_window"),
        )
        .withColumn(
            "draw",
            F.sha2(
                F.concat_ws("|", F.lit(PILOT_SEED), "native_identity", "release_group_id"),
                256,
            ),
        )
    )
    group_draw = Window.partitionBy("release_group_id").orderBy("draw", "release_row")
    source_draw = Window.partitionBy("tier", "format", "source_id").orderBy("draw", "release_row")
    format_draw = Window.partitionBy("tier", "format").orderBy("source_rank", "draw", "release_row")
    pool = (
        eligible.withColumn("group_rank", F.row_number().over(group_draw))
        .where(F.col("group_rank") == 1)
        .withColumn("source_rank", F.row_number().over(source_draw))
        .withColumn("format_rank", F.row_number().over(format_draw))
        .where(F.col("format_rank") <= 100)
        .select(
            "release_row",
            "release_group_id",
            "format",
            "source_id",
            "tier",
            "units",
            "draw",
            "format_rank",
        )
        .orderBy("format_rank", "draw", "release_row")
        .collect()
    )
    budgets = {
        "multiple_windows": 96,
        "structural_review": 48,
        "longer_single_window": 48,
        "short_control": 48,
    }
    picked = {}
    by_tier = {tier: 0 for tier in budgets}
    for tier, quota in budgets.items():
        for row in pool:
            if row.tier == tier and row.units <= quota - by_tier[tier]:
                picked[row.release_row] = row
                by_tier[tier] += row.units
                if by_tier[tier] == quota:
                    break
    remaining = PILOT_TARGET_UNITS - sum(by_tier.values())
    for row in pool:
        if remaining == 0:
            break
        if row.tier == "short_control" and row.release_row not in picked:
            picked[row.release_row] = row
            by_tier[row.tier] += row.units
            remaining -= row.units
    if remaining or len({row.release_group_id for row in picked.values()}) != len(picked):
        raise ValueError("The pilot must cover 240 units in distinct complete parent groups.")

    pilot_ids = F.col("release_row").isin(*picked)
    queued_pilot = {row.release_row: row for row in review_queue.where(pilot_ids).collect()}
    text_rows = parents.where(pilot_ids).select("release_row", "clean_text", "paragraphs").collect()
    if set(queued_pilot) != set(picked) or {row.release_row for row in text_rows} != set(picked):
        raise ValueError("A selected parent is missing from the text or review queue.")

    PACKET_INDEX_FIELDS = ("packet_sha256", "unit_ordinal", "start", "end")
    pilot_inventory = []
    for row in sorted(text_rows, key=lambda row: row.release_row):
        queued = queued_pilot[row.release_row]
        packets = make_review_packets(queued, row.clean_text, row.paragraphs)
        for packet in packets:
            pilot_inventory.append(
                {
                    "release_row": row.release_row,
                    "release_group_id": queued.release_group_id,
                    "format": queued.format,
                    "source_id": queued.source_id,
                    "tier": picked[row.release_row].tier,
                    "body_sha256": queued.clean_sha256,
                    "plan_sha256": queued.plan_sha256,
                    **{key: packet[key] for key in PACKET_INDEX_FIELDS},
                }
            )
    if (
        len(pilot_inventory) != PILOT_TARGET_UNITS
        or len({packet["packet_sha256"] for packet in pilot_inventory}) != PILOT_TARGET_UNITS
    ):
        raise ValueError("The pilot inventory contains a missing or repeated packet.")
    with tempfile.TemporaryDirectory(prefix=".review-pilot-", dir=pilot_dir.parent) as local:
        staged_pilot = Path(local) / "snapshot"
        staged_pilot.mkdir()
        file = staged_pilot / "PACKETS.json"
        file.write_text(json.dumps(pilot_inventory, indent=2) + "\n")
        pilot_manifest = {
            **pilot_contract,
            "full_parent_records": len(picked),
            "source_ids": len({row.source_id for row in picked.values()}),
            "format_count": len({row.format for row in picked.values()}),
            "units_by_tier": by_tier,
            "packets_sha256": sha256_file(file),
            "review_route_status": "prepared",
        }
        (staged_pilot / "MANIFEST.json").write_text(json.dumps(pilot_manifest, indent=2) + "\n")
        if pilot_dir.exists():
            raise FileExistsError("The pilot destination appeared during publication.")
        staged_pilot.rename(pilot_dir)

paired_requests = len(pilot_inventory) * len(PILOT_LANES)
print(
    f"Frozen {len(pilot_inventory)} packets across {pilot_manifest['full_parent_records']} "
    f"complete parents. The paired design has {paired_requests} model decisions pending."
)
display(pd.DataFrame(pilot_manifest["units_by_tier"].items(), columns=["Tier", "Units"]))

## Rehearse interrupted submission and receipt reconciliation

Run the same 480 paired packet IDs against a local fixture provider. Exit the first
process after a fixture batch is accepted and before its provider ID is saved. Restart from
the prepared input digest. Exit the second process with an open receipt transaction to
verify that only committed receipts survive. Replay all result files after resuming.
An ambiguous submission with no provider acknowledgement stays in reconciliation.
An exclusive writer lock rejects a second controller.

In production, persist the immutable request list and input checksum before dispatch.
Bind the provider batch ID by request metadata and input identity. Validate and commit
each receipt transactionally. Use the same durable ledger after a process or operator
handoff. The next worker reconciles submitted batches and takes only unfinished work.
The fixture provider exercises local durability and replay. A live route still requires
a small transport and reviewer-quality calibration.


In [ ]:
import fcntl
import sqlite3
import subprocess
import sys

# Persistent provider state and the client ledger live in separate files in this rehearsal.
worker_source = """
import fcntl
import json
import os
import sqlite3
import sys
from hashlib import sha256
from pathlib import Path

root = Path(sys.argv[1])
mode = sys.argv[2]
ownership = (root / "writer.lock").open("a")
try:
    fcntl.flock(ownership, fcntl.LOCK_EX | fcntl.LOCK_NB)
except BlockingIOError:
    sys.exit(79)
ledger = sqlite3.connect(root / "ledger.sqlite")
provider = sqlite3.connect(root / "fixture_provider.sqlite")
ledger.execute("PRAGMA synchronous=FULL")
provider.execute("PRAGMA synchronous=FULL")
jobs = json.loads((root / "jobs.json").read_text())


def exact_response(lane, packet_id):
    # Fixture receipts exercise transport only. No source text is presented or answered.
    return json.dumps({
        "lane": lane, "packet_sha256": packet_id, "outcome": "fixture",
    }, sort_keys=True)


def dispatch(job):
    key = job["batch_key"]
    state, remote_id = ledger.execute(
        "SELECT state, provider_id FROM batches WHERE batch_key=?", (key,)
    ).fetchone()
    if remote_id is not None:
        return remote_id

    matches = provider.execute(
        "SELECT provider_id FROM remote_batches WHERE batch_key=? AND input_sha=?",
        (key, job["input_sha"]),
    ).fetchall()
    if len(matches) > 1:
        raise ValueError("Multiple provider batches claim the same prepared input.")
    if matches:
        remote_id = matches[0][0]
    else:
        if state == "submitting":
            # No provider receipt yet. Preserve this ambiguous batch for reconciliation.
            return None
        with ledger:
            ledger.execute("UPDATE batches SET state='submitting' WHERE batch_key=?", (key,))
        if mode == "unknown_submit":
            os._exit(77)
        with provider:
            remote_id = provider.execute(
                "INSERT INTO remote_batches(batch_key,input_sha) VALUES(?,?)",
                (key, job["input_sha"]),
            ).lastrowid
        if mode == "after_submit":
            os._exit(75)
    with ledger:
        ledger.execute(
            "UPDATE batches SET state='submitted',provider_id=? WHERE batch_key=?",
            (remote_id, key),
        )
    return remote_id


def ingest(job, remote_id):
    remote = provider.execute(
        "SELECT batch_key,input_sha FROM remote_batches WHERE provider_id=?", (remote_id,)
    ).fetchone()
    if remote != (job["batch_key"], job["input_sha"]):
        raise ValueError("Provider result is bound to a different batch.")
    # Reverse order exercises ingest independent of result ordering.
    for index, packet_id in enumerate(reversed(job["packets"])):
        receipt = exact_response(job["lane"], packet_id)
        digest = sha256(receipt.encode()).hexdigest()
        binding = (job["lane"], packet_id)
        with ledger:
            known = ledger.execute(
                "SELECT batch_key FROM expected WHERE lane=? AND packet_sha=?", binding
            ).fetchone()
            if known != (job["batch_key"],):
                raise ValueError("Received a packet outside this batch.")
            prior = ledger.execute(
                "SELECT receipt_sha FROM receipts WHERE lane=? AND packet_sha=?", binding
            ).fetchone()
            if prior is None:
                ledger.execute(
                    "INSERT INTO receipts(lane,packet_sha,receipt_sha) VALUES(?,?,?)",
                    (*binding, digest),
                )
            elif prior != (digest,):
                raise ValueError("Duplicate receipt differs from the committed receipt.")
            if mode == "mid_ingest" and index == 11:
                os._exit(76)
    with ledger:
        ledger.execute("UPDATE batches SET state='complete' WHERE batch_key=?", (job["batch_key"],))


for job in jobs:
    remote_id = dispatch(job)
    if remote_id is None:
        sys.exit(78)
    ingest(job, remote_id)
"""


def plan_fixture_jobs():
    ids = sorted(packet["packet_sha256"] for packet in pilot_inventory)
    return [
        {
            "lane": lane,
            "packets": packets,
            "input_sha": canonical_sha256({"lane": lane, "packets": packets, **PILOT_LANES[lane]}),
            "batch_key": canonical_sha256(
                {
                    "lane": lane,
                    "packets": packets,
                    "manifest": pilot_manifest["packets_sha256"],
                }
            ),
        }
        for lane in PILOT_LANES
        for packets in (ids[:120], ids[120:])
    ]


def prepare_fixture(root, jobs):
    root.mkdir()
    (root / "worker.py").write_text(worker_source)
    (root / "jobs.json").write_text(json.dumps(jobs))
    with sqlite3.connect(root / "ledger.sqlite") as ledger:
        ledger.executescript("""
            CREATE TABLE batches (
                batch_key TEXT PRIMARY KEY, state TEXT NOT NULL, provider_id INTEGER UNIQUE
            );
            CREATE TABLE expected (
                lane TEXT, packet_sha TEXT, batch_key TEXT, PRIMARY KEY (lane,packet_sha)
            );
            CREATE TABLE receipts (
                lane TEXT, packet_sha TEXT, receipt_sha TEXT, PRIMARY KEY (lane,packet_sha)
            );
        """)
        for job in jobs:
            ledger.execute("INSERT INTO batches VALUES (?, 'prepared', NULL)", (job["batch_key"],))
            ledger.executemany(
                "INSERT INTO expected VALUES(?,?,?)",
                [(job["lane"], packet, job["batch_key"]) for packet in job["packets"]],
            )
    with sqlite3.connect(root / "fixture_provider.sqlite") as provider:
        provider.execute("""
            CREATE TABLE remote_batches (
                provider_id INTEGER PRIMARY KEY, batch_key TEXT, input_sha TEXT
            )
        """)


def read_fixture(root, database, query):
    with sqlite3.connect(root / database) as conn:
        return conn.execute(query).fetchall()


def run_fixture(root, mode, expected_exit):
    result = subprocess.run(
        [sys.executable, str(root / "worker.py"), str(root), mode],
        capture_output=True,
        text=True,
        timeout=30,
        check=False,
    )
    if result.returncode != expected_exit:
        raise RuntimeError(
            f"Unexpected rehearsal exit {result.returncode} during {mode}. {result.stderr}"
        )


fixture_results = {}
with tempfile.TemporaryDirectory(prefix="agent-review-resume-") as local:
    base = Path(local)
    jobs = plan_fixture_jobs()
    normal = base / "recovery"
    prepare_fixture(normal, jobs)
    with (normal / "writer.lock").open("a") as ownership:
        fcntl.flock(ownership, fcntl.LOCK_EX | fcntl.LOCK_NB)
        run_fixture(normal, "complete", 79)
    fixture_results["concurrent_writers_rejected"] = 1
    run_fixture(normal, "after_submit", 75)
    assert read_fixture(
        normal, "ledger.sqlite", "SELECT count(*) FROM batches WHERE state='submitting'"
    ) == [(1,)]
    assert read_fixture(
        normal, "fixture_provider.sqlite", "SELECT count(*) FROM remote_batches"
    ) == [(1,)]
    run_fixture(normal, "mid_ingest", 76)
    assert read_fixture(normal, "ledger.sqlite", "SELECT count(*) FROM receipts") == [(11,)]
    assert read_fixture(
        normal, "fixture_provider.sqlite", "SELECT count(*) FROM remote_batches"
    ) == [(1,)]
    run_fixture(normal, "complete", 0)
    assert read_fixture(
        normal, "ledger.sqlite", "SELECT count(*) FROM batches WHERE state='complete'"
    ) == [(len(jobs),)]
    before_replay = read_fixture(
        normal, "ledger.sqlite", "SELECT * FROM receipts ORDER BY lane, packet_sha"
    )
    assert len(before_replay) == paired_requests
    run_fixture(normal, "complete", 0)
    assert (
        read_fixture(normal, "ledger.sqlite", "SELECT * FROM receipts ORDER BY lane, packet_sha")
        == before_replay
    )
    remote_submissions = read_fixture(
        normal, "fixture_provider.sqlite", "SELECT count(*) FROM remote_batches"
    )[0][0]
    assert remote_submissions == len(jobs)
    fixture_results.update(
        {
            "fixture_batches": remote_submissions,
            "fixture_receipts": len(before_replay),
            "uncommitted_receipts_retained": 0,
            "duplicate_batch_submissions": remote_submissions - len(jobs),
            "duplicate_receipts_after_replay": 0,
        }
    )

    ambiguous = base / "ambiguous"
    prepare_fixture(ambiguous, jobs)
    run_fixture(ambiguous, "unknown_submit", 77)
    run_fixture(ambiguous, "complete", 78)
    assert read_fixture(
        ambiguous, "fixture_provider.sqlite", "SELECT count(*) FROM remote_batches"
    ) == [(0,)]
    fixture_results["ambiguous_submissions_held_for_reconciliation"] = 1

display(pd.DataFrame(fixture_results.items(), columns=["Local rehearsal check", "Count"]))
print(
    "Independent processes recovered the durable fixture ledger after interrupted submission "
    "and receipt ingest. The real paired pilot remains prepared."
)